# 02. Legacy LFW 압축기 전이 (Step 1 주 실험 실행 금지)

이 노트북은 과거 LFW-development 압축기를 SurvFace에 전이한 실험을 재현하기 위해 보존합니다. 현재 Step 1 주 실험은 `training_manifest.csv`의 SurvFace development에서 압축기를 fit하고 training calibration에서 threshold를 정하므로, 새 결과는 06 노트북에서 생성합니다.

| 모드 | 예상 시간 |
| --- | ---: |
| `EXECUTE_STAGE=False` | 1초 미만 |
| `EXECUTE_STAGE=True` | 약 10초~1분(artifact hash 검증/복사) |

> **진행/체크포인트/재시작**: source run의 02/03 완료 artifact hash를 검증한 뒤 attempt별 사본과 provenance JSON을 저장합니다. 중단되면 Kernel Restart 후 처음부터 재실행합니다. 동일 hash의 기존 artifact를 덮어쓰지 않고 새 attempt를 만듭니다. source run이나 모델이 바뀌면 00부터 새 SurvFace run을 만드십시오.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'dev'             # 'dev' 또는 'real'
DATA_FRACTION = 1.0     # 0 < DATA_FRACTION <= 1
SEED = 42

import sys
from pathlib import Path

for _scope_root in (Path.cwd(), *Path.cwd().parents):
    if (_scope_root / 'research').is_dir():
        break
else:
    raise FileNotFoundError('D:/ronbun 내부에서 노트북을 실행하십시오.')
if str(_scope_root) not in sys.path:
    sys.path.insert(0, str(_scope_root))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.experiments.scope import ExperimentScope

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 공통 압축 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
EXPERIMENT_SCOPE.as_dict()


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
EXECUTE_LEGACY_TRANSFER = False
if EXECUTE_STAGE or EXECUTE_LEGACY_TRANSFER:
    raise RuntimeError(
        "Step 1 주 실험은 SurvFace training development를 fit source로 사용합니다. "
        "06_step1_compression_characterization.ipynb를 실행하십시오."
    )
FIT_ON_OFFICIAL_TEST = False  # 안전장치: True로 바꿀 수 없는 정책 값
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"

def resolve_run_for_preflight():
    try:
        return resolve_active_run(
            RUN_ROOT, environment_variable="RONBUN_SURVFACE_RUN_DIR"
        ), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

RUN_DIR, RUN_RESOLUTION_ERROR = resolve_run_for_preflight()
SOURCE_LFW_RUN_DIR_VALUE = os.environ.get("RONBUN_LFW_MODEL_RUN_DIR", "").strip()
SOURCE_LFW_RUN_DIR = Path(SOURCE_LFW_RUN_DIR_VALUE).resolve() if SOURCE_LFW_RUN_DIR_VALUE else None
PROGRESS = ProgressReporter("SurvFace 02 external compressor import", heartbeat_seconds=30)


## 1. source run preflight

`RONBUN_LFW_MODEL_RUN_DIR`는 LFW development로 compressor를 fit하고 03까지 완료한 immutable provenance source를 가리켜야 합니다.


In [ ]:
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "fit_on_official_test": FIT_ON_OFFICIAL_TEST,
    "survface_run_dir": str(RUN_DIR) if RUN_DIR else None,
    "run_resolution_error": RUN_RESOLUTION_ERROR,
    "source_lfw_run_dir": str(SOURCE_LFW_RUN_DIR) if SOURCE_LFW_RUN_DIR else None,
    "source_run_manifest_exists": bool(
        SOURCE_LFW_RUN_DIR and (SOURCE_LFW_RUN_DIR / "run_manifest.json").is_file()
    ),
}
preflight


## 2. fit split·model hash·정규화 통계 동결

02 model summary의 `fit_split=development`와 LFW 03의 `error_normalization`을 모두 요구합니다. 공식 test에서 재계산한 통계는 허용하지 않습니다.


In [ ]:
def latest_completed_phase(run_dir: Path, phase_name: str) -> tuple[dict, Path]:
    attempts = run_dir / "phases" / phase_name / "attempts"
    completed = []
    for path in sorted(attempts.glob("A*/phase_manifest.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        if payload.get("status") == "completed":
            completed.append((int(payload["attempt"]), payload, path))
    if not completed:
        raise RuntimeError(f"{phase_name}의 completed attempt가 없습니다: {run_dir}")
    _, payload, path = max(completed, key=lambda item: item[0])
    return payload, path


def artifact_paths(run_dir: Path, payload: dict) -> list[Path]:
    return [run_dir / item["path"] for item in payload.get("details", {}).get("artifacts", [])]


def verify_phase_artifacts_read_only(run_dir: Path, payload: dict) -> None:
    for item in payload.get("details", {}).get("artifacts", []):
        path = run_dir / item["path"]
        if not path.is_file() or sha256_file(path) != item["sha256"]:
            raise ValueError(f"source artifact가 없거나 hash가 다릅니다: {path}")


def first_json_with(paths: list[Path], key: str) -> tuple[Path, dict]:
    for path in paths:
        if path.suffix.lower() != ".json":
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        if key in payload:
            return path, payload
    raise RuntimeError(f"{key!r}를 포함한 JSON artifact가 없습니다.")


result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    if FIT_ON_OFFICIAL_TEST:
        raise RuntimeError("SurvFace 공식 test에서 compressor/calibrator fit은 금지됩니다.")
    if RUN_DIR is None or SOURCE_LFW_RUN_DIR is None:
        raise RuntimeError("SurvFace run과 RONBUN_LFW_MODEL_RUN_DIR를 모두 지정하십시오.")

    import pandas as pd

    from research.runtime.hashing import sha256_file

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    manifest = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "survface" / "official_manifest.csv")
    if set(manifest["split"].astype(str)) != {"test"}:
        raise ValueError("SurvFace 공식 bundle은 test 행만 포함해야 합니다.")

    source_run = RunStore.open(SOURCE_LFW_RUN_DIR)
    source_manifest = json.loads(source_run.manifest_path.read_text(encoding="utf-8"))
    for entry in source_manifest.get("inputs", []):
        source_input = Path(str(entry["path"]))
        if not source_input.is_file() or sha256_file(source_input) != entry["sha256"]:
            raise ValueError(f"source run input이 없거나 hash가 다릅니다: {source_input}")
    fit_phase, _ = latest_completed_phase(SOURCE_LFW_RUN_DIR, "02_compressor_fit")
    material_phase, _ = latest_completed_phase(
        SOURCE_LFW_RUN_DIR, "03_compressed_materialization_and_index"
    )
    verify_phase_artifacts_read_only(SOURCE_LFW_RUN_DIR, fit_phase)
    verify_phase_artifacts_read_only(SOURCE_LFW_RUN_DIR, material_phase)
    fit_artifacts = artifact_paths(SOURCE_LFW_RUN_DIR, fit_phase)
    material_artifacts = artifact_paths(SOURCE_LFW_RUN_DIR, material_phase)
    pca_path = next((p for p in fit_artifacts if p.suffix.lower() == ".joblib"), None)
    pq_path = next((p for p in fit_artifacts if p.suffix.lower() == ".faiss"), None)
    if pca_path is None or pq_path is None:
        raise RuntimeError("LFW 02에서 PCA .joblib과 PQ .faiss를 모두 찾지 못했습니다.")
    fit_summary_path, fit_summary = first_json_with(fit_artifacts, "fit_split")
    error_summary_path, error_summary = first_json_with(material_artifacts, "error_normalization")
    if str(fit_summary["fit_split"]) != "development":
        raise ValueError(f"compressor fit_split이 development가 아닙니다: {fit_summary['fit_split']}")
    source_manifest_path = str(source_manifest.get("config", {}).get("dataset", {}).get("manifest_path", ""))
    if "lfw" not in source_manifest_path.lower():
        raise ValueError(f"source run이 LFW manifest를 가리키지 않습니다: {source_manifest_path}")

    source_inputs = {
        "lfw_source_run_manifest": source_run.manifest_path,
        "lfw_pca_model": pca_path,
        "lfw_pq_model": pq_path,
        "lfw_fit_summary": fit_summary_path,
        "lfw_error_normalization": error_summary_path,
    }
    for role, path in source_inputs.items():
        run.record_input(path, role=role)

    with run.phase("02_external_compressor_import") as phase:
        suffix = f"A{phase.attempt:03d}"
        published_pca = phase.publish_artifact(pca_path, name=f"lfw_pca_{suffix}.joblib")
        published_pq = phase.publish_artifact(pq_path, name=f"lfw_pq_{suffix}.faiss")
        transfer = {
            "source_dataset": "lfw",
            "target_dataset": "qmul-survface-v1",
            "source_run_id": source_run.run_id,
            "source_run_dir": str(SOURCE_LFW_RUN_DIR),
            "fit_split": "development",
            "fit_count": int(fit_summary["fit_count"]),
            "fit_on_survface_official_test": False,
            "pca_artifact": str(published_pca.relative_to(run.run_dir)),
            "pca_sha256": sha256_file(published_pca),
            "pq_artifact": str(published_pq.relative_to(run.run_dir)),
            "pq_sha256": sha256_file(published_pq),
            "error_normalization": error_summary["error_normalization"],
            "error_normalization_source": str(error_summary_path),
        }
        transfer_path = phase.attempt_dir / f"external_compressor_manifest_{suffix}.json"
        transfer_path.write_text(json.dumps(transfer, ensure_ascii=False, indent=2), encoding="utf-8")
        phase.publish_artifact(transfer_path)
        phase.record_counts(imported_models=2, fit_count=transfer["fit_count"])
        phase.record("cross_dataset_transfer", **transfer)
    PROGRESS.emit("02 완료", source_run_id=source_run.run_id, fit_split="development")
    result = {"status": "completed", "run_id": run.run_id, **transfer}
else:
    PROGRESS.emit("검토 모드 완료: 모델을 fit/import하지 않음", expected="1초 미만")
result


## 다음 단계

`fit_split=development`, `fit_on_survface_official_test=False`, 두 모델 hash, frozen `error_normalization`을 확인합니다. 이 조건 중 하나라도 없으면 03으로 진행하지 않습니다.
